# C05 Logit Adjustment tuning

Train with class-prior logit adjustment and evaluate with the same adjustment applied at prediction time.


In [ ]:
import json, random, numpy as np, pandas as pd, torch, joblib
import torch.nn.functional as F
from pathlib import Path
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


In [ ]:
CWD = Path.cwd()
NOTEBOOKS_DIR = CWD.parent if CWD.name == "challengers" else CWD
PROJECT_ROOT = NOTEBOOKS_DIR.parent
DATA_DIR = PROJECT_ROOT / "data" / "processed"
MODELS_DIR = PROJECT_ROOT / "notebooks" / "models" / "challengers"
FIGURES_DIR = PROJECT_ROOT / "figures" / "challengers"
MODELS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print(f"CWD: {CWD}")
print(f"NOTEBOOKS_DIR: {NOTEBOOKS_DIR}")
print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"DATA_DIR exists: {(DATA_DIR / 'train.csv').exists()}")
print(f"MODELS_DIR: {MODELS_DIR}")
print(f"FIGURES_DIR: {FIGURES_DIR}")


In [ ]:
df_train = pd.read_csv(DATA_DIR / "train.csv")
df_val = pd.read_csv(DATA_DIR / "val.csv")
df_test = pd.read_csv(DATA_DIR / "test.csv")

mapping = pd.read_csv(DATA_DIR / "label_to_supercategory_v1.csv")
label_to_supercat = dict(zip(mapping["label"], mapping["supercategory"]))
for df in [df_train, df_val, df_test]:
    df["supercategory"] = df["label"].map(label_to_supercat)

le = LabelEncoder()
df_train["y"] = le.fit_transform(df_train["supercategory"])
df_val["y"] = le.transform(df_val["supercategory"])
df_test["y"] = le.transform(df_test["supercategory"])
num_labels = len(le.classes_)

city_counts = df_train["city_group"].value_counts()
raw_w = 1.0 / np.sqrt(city_counts)
city_weight_map = (raw_w / raw_w.mean()).to_dict()
df_train["sample_weight"] = df_train["city_group"].map(city_weight_map).astype(float)

class_counts = df_train["y"].value_counts().sort_index().to_numpy()
class_priors = class_counts / class_counts.sum()
log_priors = np.log(np.clip(class_priors, 1e-12, None)).astype(np.float32)
print(f"Train: {len(df_train)}, Val: {len(df_val)}, Test: {len(df_test)}")
print(f"Labels: {num_labels}, City groups: {len(city_counts)}")
print("Class priors:", class_priors.round(6).tolist())


In [ ]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

def tokenize(batch):
    return tokenizer(batch["resume_text"], padding="max_length", truncation=True, max_length=128)

train_ds = Dataset.from_pandas(df_train[["resume_text", "y", "sample_weight"]]).map(tokenize, batched=True)
val_ds = Dataset.from_pandas(df_val[["resume_text", "y"]]).map(tokenize, batched=True)
test_ds = Dataset.from_pandas(df_test[["resume_text", "y"]]).map(tokenize, batched=True)

train_ds = train_ds.rename_column("y", "labels")
val_ds = val_ds.rename_column("y", "labels")
test_ds = test_ds.rename_column("y", "labels")

train_ds.set_format("torch", columns=["input_ids", "attention_mask", "labels", "sample_weight"])
val_ds.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
test_ds.set_format("torch", columns=["input_ids", "attention_mask", "labels"])


In [ ]:
class LogitAdjustedTrainer(Trainer):
    def __init__(self, *args, log_priors, tau=1.0, **kwargs):
        super().__init__(*args, **kwargs)
        self.log_priors = log_priors
        self.tau = tau
        print(f"Logit Adjustment: tau={tau}")

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        sample_weight = inputs.pop("sample_weight", None)
        labels = inputs["labels"]
        outputs = model(**inputs)
        logits = outputs.logits
        adjusted_logits = logits + self.tau * self.log_priors.to(logits.device)

        per_sample_loss = F.cross_entropy(adjusted_logits, labels, reduction="none")
        if sample_weight is not None:
            per_sample_loss = per_sample_loss * sample_weight.to(per_sample_loss.dtype)

        loss = per_sample_loss.mean()
        return (loss, outputs) if return_outputs else loss


def compute_metrics(prediction_output):
    preds = np.argmax(prediction_output.predictions, axis=1)
    return {
        "accuracy": accuracy_score(prediction_output.label_ids, preds),
        "macro_f1": f1_score(prediction_output.label_ids, preds, average="macro"),
    }


def ovr_rates(df, group_col, num_classes):
    groups = sorted(df[group_col].dropna().unique())
    tpr = np.zeros((len(groups), num_classes))
    support = np.zeros((len(groups), num_classes))
    for gi, group_name in enumerate(groups):
        dg = df[df[group_col] == group_name]
        yt, yp = dg["y_true"].values, dg["y_pred"].values
        for c in range(num_classes):
            positive_mask = yt == c
            tp = np.sum((yp == c) & positive_mask)
            fn = np.sum((yp != c) & positive_mask)
            support[gi, c] = positive_mask.sum()
            tpr[gi, c] = tp / (tp + fn) if (tp + fn) > 0 else np.nan
    return tpr, support


def robust_gaps(tpr, support, min_support=30):
    gaps = []
    for c in range(tpr.shape[1]):
        col = tpr[support[:, c] >= min_support, c]
        col = col[~np.isnan(col)]
        gaps.append(col.max() - col.min() if len(col) >= 2 else np.nan)
    gaps = np.array(gaps)
    valid = gaps[~np.isnan(gaps)]
    return (valid.max() if len(valid) else np.nan, valid.mean() if len(valid) else np.nan)


In [ ]:
RUNS = [
    {"tag": "tau05_2ep", "tau": 0.5, "epochs": 2},
    {"tag": "tau10_2ep", "tau": 1.0, "epochs": 2},
    {"tag": "tau15_2ep", "tau": 1.5, "epochs": 2},
]

summary_rows = []
base_log_priors = torch.tensor(log_priors, dtype=torch.float32)

for run in RUNS:
    model_name = f"logit_adjustment_{run['tag']}"
    save_dir = MODELS_DIR / model_name
    checkpoint_dir = save_dir / "checkpoints"

    print()
    print("=" * 80)
    print(f"Training {model_name}: tau={run['tau']}, epochs={run['epochs']}")

    model = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=num_labels)
    args = TrainingArguments(
        output_dir=str(checkpoint_dir),
        learning_rate=2e-5,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=8,
        num_train_epochs=run["epochs"],
        weight_decay=0.01,
        warmup_ratio=0.1,
        logging_steps=100,
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=1,
        load_best_model_at_end=True,
        metric_for_best_model="macro_f1",
        greater_is_better=True,
        remove_unused_columns=False,
        report_to="none",
        seed=SEED,
    )

    trainer = LogitAdjustedTrainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        compute_metrics=compute_metrics,
        log_priors=base_log_priors,
        tau=run["tau"],
    )

    trainer.train()
    pred_output = trainer.predict(test_ds)
    y_true = pred_output.label_ids
    adjusted_logits = pred_output.predictions + run["tau"] * log_priors[None, :]
    y_pred = np.argmax(adjusted_logits, axis=1)

    acc = float(accuracy_score(y_true, y_pred))
    macro_f1 = float(f1_score(y_true, y_pred, average="macro"))

    df_eval = df_test.copy()
    df_eval["y_true"] = y_true
    df_eval["y_pred"] = y_pred
    tpr, support = ovr_rates(df_eval, "city_group", num_labels)
    worst_gap, macro_gap = robust_gaps(tpr, support, min_support=30)

    save_dir.mkdir(parents=True, exist_ok=True)
    trainer.model.save_pretrained(save_dir, safe_serialization=True)
    tokenizer.save_pretrained(save_dir)
    joblib.dump(le, save_dir / "label_encoder.joblib")

    config = {
        "method": "Logit Adjustment + sqrt_rw",
        "tau": run["tau"],
        "epochs": run["epochs"],
        "learning_rate": 2e-5,
        "weight_decay": 0.01,
        "warmup_ratio": 0.1,
        "class_priors": class_priors.tolist(),
        "log_priors": log_priors.tolist(),
        "accuracy": acc,
        "macro_f1": macro_f1,
        "tpr_gap_worst_robust": float(worst_gap),
        "tpr_gap_macro_robust": float(macro_gap),
    }
    with open(save_dir / "training_config.json", "w") as f:
        json.dump(config, f, indent=2)

    summary_rows.append({
        "model_name": model_name,
        "tau": run["tau"],
        "epochs": run["epochs"],
        "accuracy": acc,
        "macro_f1": macro_f1,
        "worst_gap": float(worst_gap),
        "macro_gap": float(macro_gap),
        "model_dir": str(save_dir.relative_to(PROJECT_ROOT)),
    })

summary_df = pd.DataFrame(summary_rows).sort_values(["worst_gap", "macro_f1"], ascending=[True, False]).reset_index(drop=True)
summary_path = FIGURES_DIR / "c05_logit_adjustment_tuning_summary.csv"
summary_df.to_csv(summary_path, index=False)
summary_df


In [ ]:
print(f"Summary saved to: {summary_path}")
summary_df
